In [1]:
import os
import re
import json
import string
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import clone_model
from tensorflow.keras.applications import EfficientNetV2B0
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, TFAutoModel, AutoTokenizer, BertTokenizer
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split

2025-08-16 14:09:15.987942: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = tf.data.Dataset.load(
    "dataset",
    element_spec=({
        'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
    }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
)

dataVal = tf.data.Dataset.load(
    "dataVal",
    element_spec=({
        'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
    }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
)

2025-08-16 14:09:18.669758: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-16 14:09:18.678907: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-16 14:09:18.678962: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-16 14:09:18.681383: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-16 14:09:18.681434: I external/local_xla/xla/stream_executor

In [3]:
class BertEmbeddingLayer(tf.keras.layers.Layer):
    def __init__(self, model_name, **kwargs):
        super(BertEmbeddingLayer, self).__init__(**kwargs)
        self.model_name = model_name
        self.embedding_size = None
        self.bert = None

    def build(self, input_shape):
        self.bert = TFAutoModel.from_pretrained(self.model_name)
        self.bert.trainable = True
        self.embedding_size = self.bert.config.hidden_size
        super(BertEmbeddingLayer, self).build(input_shape)

    def call(self, inputs):
        ids, att = inputs
        outputs = self.bert(input_ids=ids, attention_mask=att)
        return outputs.pooler_output

    def get_config(self):
        config = super(BertEmbeddingLayer, self).get_config()
        config.update({
            "model_name": self.model_name
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        clean_config = {k: v for k, v in config.items() 
                      if k in ['model_name', 'name', 'trainable', 'dtype']}
        return cls(**clean_config)

In [4]:
def makemodel(output_bias=None):
    # Text using IndoBertp1
    ids1 = tf.keras.layers.Input(shape=(32,), dtype=tf.int32, name="title")
    att1 = tf.keras.layers.Input(shape=(32,), dtype=tf.int32, name="titlemask")
    ids2 = tf.keras.layers.Input(shape=(128,), dtype=tf.int32, name="content")
    att2 = tf.keras.layers.Input(shape=(128,), dtype=tf.int32, name="contentmask")

    indobert1 = BertEmbeddingLayer(model_name="indobenchmark/indobert-base-p1", name="indobert1")
    indobert2 = BertEmbeddingLayer(model_name="indobenchmark/indobert-base-p1", name="indobert2")

    title = indobert1([ids1, att1])
    content = indobert2([ids2, att2])

    # Image using EfficientNetV2s
    img1 = tf.keras.layers.Input(shape=(224,224,3), dtype=tf.uint8, name="image")

    effi = EfficientNetV2B0(
        include_top = False,
        weights = 'imagenet',
        input_shape = (224,224,3),
        pooling = 'max'
    )
    image = effi(img1)

    fin = tf.keras.layers.concatenate([title, content, image])
    fin = tf.keras.layers.BatchNormalization()(fin)
        
    fin = tf.keras.layers.Dense(512, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 

    fin = tf.keras.layers.Dense(256, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 
    
    fin = tf.keras.layers.Dense(128, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 
    
    fin = tf.keras.layers.Dense(64, activation='relu')(fin)
    fin = tf.keras.layers.Dropout(0.2)(fin)
    fin = tf.keras.layers.Dense(1, activation='sigmoid')(fin)

    final = tf.keras.Model(inputs=[ids1, att1, ids2, att2, img1], outputs=fin)

    for layer in final.layers:
      layer.trainable = True
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.00005)
    final.compile(
        loss = tf.keras.losses.BinaryCrossentropy(),
        optimizer=optimizer,
        metrics=[tf.keras.metrics.BinaryAccuracy(),tf.keras.metrics.Precision(),tf.keras.metrics.Recall()]
    )

    return final

final = makemodel()
final.summary()

Some layers from the model checkpoint at indobenchmark/indobert-base-p1 were not used when initializing TFBertModel: ['nsp___cls', 'mlm___cls', 'bert/encoder/layer_._2/attention/output/LayerNorm/gamma:0', 'bert/encoder/layer_._7/attention/self/value/kernel:0', 'bert/encoder/layer_._10/attention/self/key/kernel:0', 'bert/encoder/layer_._5/attention/self/value/kernel:0', 'bert/encoder/layer_._10/output/dense/bias:0', 'bert/encoder/layer_._3/output/LayerNorm/gamma:0', 'bert/encoder/layer_._11/attention/output/dense/kernel:0', 'bert/encoder/layer_._10/output/LayerNorm/beta:0', 'bert/encoder/layer_._3/attention/self/key/kernel:0', 'bert/encoder/layer_._5/attention/output/dense/kernel:0', 'bert/encoder/layer_._6/attention/self/value/kernel:0', 'bert/encoder/layer_._8/attention/self/key/kernel:0', 'bert/encoder/layer_._5/output/LayerNorm/gamma:0', 'bert/encoder/layer_._1/attention/self/query/kernel:0', 'bert/encoder/layer_._2/intermediate/dense/bias:0', 'bert/encoder/layer_._2/attention/self/

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ title (InputLayer)  │ (None, 32)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ titlemask           │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content             │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ contentmask         │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ indobert1           │ (None, 768)       │          0 │ title[0][0],      │
│ (BertEmbeddingLaye… │                   │            │ titlemask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ indobert2           │ (None, 768)       │          0 │ content[0][0],    │
│ (BertEmbeddingLaye… │                   │            │ contentmask[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-b0   │ (None, 1280)      │  5,919,312 │ image[0][0]       │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 2816)      │          0 │ indobert1[0][0],  │
│ (Concatenate)       │                   │            │ indobert2[0][0],  │
│                     │                   │            │ efficientnetv2-b… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2816)      │     11,264 │ concatenate[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,442,304 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 512)       │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 256)       │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ activation_1[0][

 Total params: 7,549,009 (28.80 MB)

 Trainable params: 7,480,977 (28.54 MB)

 Non-trainable params: 68,032 (265.75 KB)

In [5]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
    )

# model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
#     filepath='./models/best_model_val_loss_{val_loss:.4f}.weights.h5',
#     monitor='val_loss',
#     save_best_only=True,
#     save_weights_only=True
# )

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=1,
    min_lr=1e-8
)

history=final.fit(x = dataset,
                  epochs = 50,
                  callbacks=[early_stopping, 
                             # model_checkpoint, 
                             reduce_lr],
                  validation_data=dataVal
                  )

Epoch 1/50


I0000 00:00:1755328225.648566   81846 service.cc:145] XLA service 0x732ce40161c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1755328225.648621   81846 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070, Compute Capability 8.6
2025-08-16 14:10:27.134436: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1755328228.005260   81846 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755328228.104242   81846 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert
2025-08-16 14:10:32.965056: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1755328250.246507   82062 asm_compiler.cc:369] ptxas warning :

349/350 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - binary_accuracy: 0.5171 - loss: 0.7887 - precision: 0.4597 - recall: 0.6232

W0000 00:00:1755328341.080736   81841 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755328341.085897   81841 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert
I0000 00:00:1755328362.436398   82160 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_7', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1755328362.459070   82158 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_31', 20 bytes spill stores, 20 bytes spill loads

I0000 00:00:1755328362.459111   82171 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_31', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1755328362.578380   82152 asm_compiler.cc:369] ptxas warning : Registers are spilled to l

350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 295ms/step - binary_accuracy: 0.5171 - loss: 0.7886 - precision: 0.4597 - recall: 0.6230

W0000 00:00:1755328403.355373   81846 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755328403.426544   81846 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert
I0000 00:00:1755328405.688449   82400 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_13839', 8 bytes spill stores, 8 bytes spill loads

W0000 00:00:1755328420.847122   81839 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755328420.851940   81839 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert
I0000 00:00:1755328422.461147   82495 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_26', 12 bytes spi

350/350 ━━━━━━━━━━━━━━━━━━━━ 252s 391ms/step - binary_accuracy: 0.5172 - loss: 0.7885 - precision: 0.4598 - recall: 0.6228 - val_binary_accuracy: 0.6349 - val_loss: 0.6479 - val_precision: 0.6343 - val_recall: 0.3356 - learning_rate: 5.0000e-05
Epoch 2/50
350/350 ━━━━━━━━━━━━━━━━━━━━ 55s 158ms/step - binary_accuracy: 0.6007 - loss: 0.6852 - precision: 0.5462 - recall: 0.4659 - val_binary_accuracy: 0.6768 - val_loss: 0.6016 - val_precision: 0.6787 - val_recall: 0.4567 - learning_rate: 5.0000e-05
Epoch 3/50
350/350 ━━━━━━━━━━━━━━━━━━━━ 55s 156ms/step - binary_accuracy: 0.6371 - loss: 0.6464 - precision: 0.5942 - recall: 0.5128 - val_binary_accuracy: 0.7015 - val_loss: 0.5763 - val_precision: 0.7029 - val_recall: 0.5172 - learning_rate: 5.0000e-05
Epoch 4/50
350/350 ━━━━━━━━━━━━━━━━━━━━ 55s 158ms/step - binary_accuracy: 0.6709 - loss: 0.6102 - precision: 0.6388 - recall: 0.5539 - val_binary_accuracy: 0.7187 - val_loss: 0.5590 - val_precision: 0.7193 - val_recall: 0.5559 - learning_rate: 5

# Randomized data creation

In [6]:
df = pd.read_csv('click-id/dataset_with_image/combined_final_final.csv')

In [7]:
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

In [8]:
df_train, df_val = train_test_split(df[['title','content','image_path_processed','label_score']],test_size = 0.2 ,random_state = 93)
df_train.shape , df_val.shape

((11174, 4), (2794, 4))

In [23]:
df_val_rand_title = df_val.copy()
df_val_rand_title['title'] = np.random.RandomState(seed=42).permutation(df_val['title'])

df_val_rand_content = df_val.copy()
df_val_rand_content['content'] = np.random.RandomState(seed=42).permutation(df_val['content'])

df_val_rand_image = df_val.copy()
df_val_rand_image['image_path_processed'] = np.random.RandomState(seed=42).permutation(df_val['image_path_processed'])

In [24]:
tokens_val_rand_title = {
  'title' : tokenizer.batch_encode_plus(
    df_val_rand_title['title'].tolist(),
    max_length = 32,
    padding = 'max_length',
    truncation=True
  ),
  'content' : tokenizer.batch_encode_plus(
    df_val_rand_title['content'].tolist(),
    max_length = 128,
    padding = 'max_length',
    truncation=True
  )
}

tokens_val_rand_content = {
  'title' : tokenizer.batch_encode_plus(
    df_val_rand_content['title'].tolist(),
    max_length = 32,
    padding = 'max_length',
    truncation=True
  ),
  'content' : tokenizer.batch_encode_plus(
    df_val_rand_content['content'].tolist(),
    max_length = 128,
    padding = 'max_length',
    truncation=True
  )
}

tokens_val_rand_image = {
  'title' : tokenizer.batch_encode_plus(
    df_val_rand_image['title'].tolist(),
    max_length = 32,
    padding = 'max_length',
    truncation=True
  ),
  'content' : tokenizer.batch_encode_plus(
    df_val_rand_image['content'].tolist(),
    max_length = 128,
    padding = 'max_length',
    truncation=True
  )
}

In [27]:
val_rand_title_images = tf.data.Dataset.load("val_rand_title_images_dataset")
val_rand_content_images = tf.data.Dataset.load("val_rand_content_images_dataset")
val_rand_image_images = tf.data.Dataset.load("val_rand_image_images_dataset")

In [28]:
validation_rand_title_data = {
    'title':tf.convert_to_tensor(tokens_val_rand_title['title']['input_ids']),
    'content':tf.convert_to_tensor(tokens_val_rand_title['content']['input_ids']),
    'titlemask':tf.convert_to_tensor(tokens_val_rand_title['title']['attention_mask']),
    'contentmask':tf.convert_to_tensor(tokens_val_rand_title['content']['attention_mask']),
}
validation_rand_content_data = {
    'title':tf.convert_to_tensor(tokens_val_rand_content['title']['input_ids']),
    'content':tf.convert_to_tensor(tokens_val_rand_content['content']['input_ids']),
    'titlemask':tf.convert_to_tensor(tokens_val_rand_content['title']['attention_mask']),
    'contentmask':tf.convert_to_tensor(tokens_val_rand_content['content']['attention_mask']),
}
validation_rand_image_data = {
    'title':tf.convert_to_tensor(tokens_val_rand_image['title']['input_ids']),
    'content':tf.convert_to_tensor(tokens_val_rand_image['content']['input_ids']),
    'titlemask':tf.convert_to_tensor(tokens_val_rand_image['title']['attention_mask']),
    'contentmask':tf.convert_to_tensor(tokens_val_rand_image['content']['attention_mask']),
}

label_val_rand_title = tf.convert_to_tensor(df_val_rand_title['label_score'].tolist())
label_val_rand_content = tf.convert_to_tensor(df_val_rand_content['label_score'].tolist())
label_val_rand_image = tf.convert_to_tensor(df_val_rand_image['label_score'].tolist())

In [29]:
dataVal_rand_title = (tf.data.Dataset.from_tensor_slices((dict(validation_rand_title_data),label_val_rand_title))
    .prefetch(tf.data.AUTOTUNE)
    .cache()
    )
dataVal_rand_title = tf.data.Dataset.zip((dataVal_rand_title, val_rand_title_images))

dataVal_rand_content = (tf.data.Dataset.from_tensor_slices((dict(validation_rand_content_data),label_val_rand_content))
    .prefetch(tf.data.AUTOTUNE)
    .cache()
    )
dataVal_rand_content = tf.data.Dataset.zip((dataVal_rand_content, val_rand_content_images))

dataVal_rand_image = (tf.data.Dataset.from_tensor_slices((dict(validation_rand_image_data),label_val_rand_image))
    .prefetch(tf.data.AUTOTUNE)
    .cache()
    )
dataVal_rand_image = tf.data.Dataset.zip((dataVal_rand_image, val_rand_image_images))

def add_image_and_fix_shapes(text_data_label, image):
    text_data, label = text_data_label
    new_data = {
        'title': tf.ensure_shape(text_data['title'], [None, 32]),
        'content': tf.ensure_shape(text_data['content'], [None, 128]),
        'titlemask': tf.ensure_shape(text_data['titlemask'], [None, 32]),
        'contentmask': tf.ensure_shape(text_data['contentmask'], [None, 128]),
        'image': image
    }
    
    return new_data, label

dataVal_rand_title = (dataVal_rand_title
    .batch(32)
    .map(add_image_and_fix_shapes)
    .prefetch(tf.data.AUTOTUNE)
    .cache()
)


dataVal_rand_content = (dataVal_rand_content
    .batch(32)
    .map(add_image_and_fix_shapes)
    .prefetch(tf.data.AUTOTUNE)
    .cache()
)


dataVal_rand_image = (dataVal_rand_image
    .batch(32)
    .map(add_image_and_fix_shapes)
    .prefetch(tf.data.AUTOTUNE)
    .cache()
)


# after data prep

In [31]:
# dataVal_rand_title = tf.data.Dataset.load(
#     "dataVal_rand_title",
#     element_spec=({
#         'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
#         'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
#         'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
#         'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
#         'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
#     }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
# )

# dataVal_rand_content = tf.data.Dataset.load(
#     "dataVal_rand_content",
#     element_spec=({
#         'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
#         'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
#         'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
#         'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
#         'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
#     }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
# )

# dataVal_rand_image = tf.data.Dataset.load(
#     "dataVal_rand_image",
#     element_spec=({
#         'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
#         'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
#         'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
#         'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
#         'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
#     }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
# )

In [32]:
y_pred = final.predict(dataVal_rand_title)
y_pred_binary = (y_pred > 0.5).astype(int)
from sklearn.metrics import classification_report
y_val = np.concatenate([y for _, y in dataVal_rand_title.as_numpy_iterator()])
print(classification_report(y_val, y_pred_binary, 
                           target_names=['Class_0', 'Class_1'],
                           digits = 4))

88/88 ━━━━━━━━━━━━━━━━━━━━ 8s 90ms/step
              precision    recall  f1-score   support

     Class_0     0.7101    0.7782    0.7426      1605
     Class_1     0.6560    0.5711    0.6106      1189

    accuracy                         0.6901      2794
   macro avg     0.6831    0.6746    0.6766      2794
weighted avg     0.6871    0.6901    0.6864      2794



2025-08-16 14:32:22.709026: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [33]:
y_pred = final.predict(dataVal_rand_content)
y_pred_binary = (y_pred > 0.5).astype(int)
from sklearn.metrics import classification_report
y_val = np.concatenate([y for _, y in dataVal_rand_content.as_numpy_iterator()])
print(classification_report(y_val, y_pred_binary, 
                           target_names=['Class_0', 'Class_1'],
                           digits = 4))

88/88 ━━━━━━━━━━━━━━━━━━━━ 10s 114ms/step
              precision    recall  f1-score   support

     Class_0     0.7231    0.8037    0.7613      1605
     Class_1     0.6881    0.5845    0.6321      1189

    accuracy                         0.7105      2794
   macro avg     0.7056    0.6941    0.6967      2794
weighted avg     0.7082    0.7105    0.7063      2794



2025-08-16 14:32:33.018713: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [34]:
y_pred = final.predict(dataVal_rand_image)
y_pred_binary = (y_pred > 0.5).astype(int)
from sklearn.metrics import classification_report
y_val = np.concatenate([y for _, y in dataVal_rand_image.as_numpy_iterator()])
print(classification_report(y_val, y_pred_binary, 
                           target_names=['Class_0', 'Class_1'],
                           digits = 4))

88/88 ━━━━━━━━━━━━━━━━━━━━ 8s 88ms/step
              precision    recall  f1-score   support

     Class_0     0.6764    0.7333    0.7037      1605
     Class_1     0.5939    0.5265    0.5582      1189

    accuracy                         0.6453      2794
   macro avg     0.6352    0.6299    0.6310      2794
weighted avg     0.6413    0.6453    0.6418      2794



2025-08-16 14:32:41.200078: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
